<a href="https://colab.research.google.com/github/Mohamed-Mohamed-Ibrahim/Code-Generation-and-Guarding/blob/SFT/sft/peft-sft-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt install tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).


In [ ]:
!pip install peft transformers[torch] trl bitsandbytes datasets -U --q

In [ ]:
import os
import random
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training

In [ ]:
dataset = load_dataset("flytech/python-codes-25k",split="train")

In [ ]:
dataset

In [ ]:
shuffled_dataset = dataset.shuffle(seed=42)
train_dataset = shuffled_dataset.select(range(4000))
eval_dataset = shuffled_dataset.select(range(200))
test_dataset = shuffled_dataset.select(range(200))

In [ ]:
print("="*100)
print(shuffled_dataset)
print("="*100)
print(shuffled_dataset[0]['input'])
print("="*100)
print(shuffled_dataset[0]['instruction'])
print("="*100)
print(shuffled_dataset[0]['text'])
print("="*100)
print(shuffled_dataset[0]['output'])
print("="*100)


In [ ]:
# HF hub ID
BASE_MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
BASE_MODEL_NAME = "arnir0/Tiny-LLM"

# Push adapter artifacts post training
OUTPUT_DIR = "./qlora-peft-output"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "adapter")

## Hyperparameters

In [ ]:
lora_r = 16
lora_target_modules = "all-linear"
bs =  1
gradient_accumulation_steps = 4
epochs = 1
lr = 2e-4
optim = "paged_adamw_8bit"
max_length = 1024

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

In [ ]:
lora_config = LoraConfig(
    r=32,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Llama-style
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
from trl import SFTConfig, SFTTrainer

# Define ALL configuration parameters within SFTConfig
config = SFTConfig(
    # --- Standard TrainingArguments parameters ---
    output_dir=OUTPUT_DIR,
    learning_rate=2e-4,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",

    # --- SFT-specific parameters (moved here from previous iterations) ---
    dataset_text_field="text",
    max_length=512,
    packing=False,
)

# Initialize the SFTTrainer, passing the SFTConfig object to the 'args' parameter
trainer = SFTTrainer(
    model=model,
    args=config,  # Pass the combined SFTConfig object here
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    # No other config parameters needed here
)

In [ ]:
trainer.train()

In [ ]:
ADAPTER_DIR = "./qlora-peft-output/adapter"

In [ ]:
# -----------------------------
# 6. Save adapter weights
# -----------------------------
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to: {ADAPTER_DIR}")

In [ ]:
!tree


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM

# -------------------------------------------------------
# Paths
# -------------------------------------------------------
BASE_ID = "meta-llama/Llama-3.1-8B-Instruct"
BASE_ID = "arnir0/Tiny-LLM"
BASE_ID = "Qwen/Qwen2-1.5B-Instruct"
ADAPTER_DIR = "./qlora-peft-output/adapter"
MERGED_DIR = "./merged-model"

# -------------------------------------------------------
# Load tokenizer
# -------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(
    BASE_ID,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# -------------------------------------------------------
# Load base model in fp16/bf16
# (merged model will be full precision LLM)
# -------------------------------------------------------
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# -------------------------------------------------------
# Attach adapter
# -------------------------------------------------------
print("Loading adapter onto base model...")
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
)

# -------------------------------------------------------
# Merge adapter → base model
# -------------------------------------------------------
print("Merging LoRA adapter into base model weights...")
merged_model = model.merge_and_unload()   # <-- key line

# -------------------------------------------------------
# Save merged full model (optional)
# -------------------------------------------------------
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print(f"\nMerged model saved to: {MERGED_DIR}\n")

In [ ]:
!tree

In [ ]:
def generate(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(merged_model.device)
    with torch.no_grad():
        outputs = merged_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# -------------------------------------------------------
# Run merged-model inference
# -------------------------------------------------------
print("==================== MERGED MODEL OUTPUT ====================\n")

for p in test_dataset:
    wrapped = (
        "You are a personalized assistant that knows details about the user based "
        "on prior fine-tuning data.\n\n"
        f"Question: {p['instruction']}\nAnswer:"
    )

    print(f"PROMPT: {p}\n")
    output = generate(wrapped)
    print(output)
    print("-" * 120)